# 01. Adaptive Compute: Early Exit, Token Merging (ToMe), and FrugalGPT Cascades

- **Category:** 15_compression_peft_and_speculative_inference / 02_adaptive_kv_speculative_and_merging
- **Lineage & Citations:** Xin et al. (ACL 2020, *DeeBERT*); Bolya et al. (ICLR 2023, *Token Merging: Your ViT but Faster*); Chen et al. (NeurIPS 2023, *FrugalGPT: Harnessing the Cost and Quality of Multiple LLMs*).
- **Purpose:** Implement dynamic inference allocation across three fundamental axes: depth-wise early exit classifiers, sequence-wise bipartite token merging without retraining, and model-wise cost-aware cascade routers.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Knowledge Distillation Logit Feature and Minillm](../01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.ipynb) $\to$ `🔗 Downstream Syntheses:` [KV Cache Paged Radix H2O Snapkv and Yoco](02_kv_cache_paged_radix_h2o_snapkv_and_yoco.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use early exit in strict real-time batching environments where divergent exit depths cause thread warp divergence or GPU underutilization without async request queues. Do not use unweighted token merging on fine-grained pixel or character tokens without attention size compensation.

## Mathematical Foundations & Governing Equations

### 1. Depth-Wise Early Exit
$$p^{(l)} = \text{softmax}(h^{(l)} W_{\text{exit}} + b_{\text{exit}}), \quad \text{exit} = \mathbb{I}(\max_c p_c^{(l)} \ge \tau)$$

### 2. Sequence-Wise Bipartite Token Merging (ToMe)
Partition tokens into disjoint sets $A$ (even) and $B$ (odd). For the top $r$ highest cosine similarity pairs $(a_i, b_j)$:
$$b_j' = \frac{s_b b_j + s_a a_i}{s_b + s_a}, \quad s_b' = s_b + s_a$$

### 3. Model-Wise FrugalGPT Cascade Routing
Sequential escalation across tiers $\mathcal{M} = [M_1, \dots, M_K]$ with cost $C_k$ and confidence $c_k$:
$$\text{Route}(x) = \min \{ k : c_k(x) \ge \tau_k \text{ or } k = K \}, \quad \mathbb{E}[\text{Cost}] = \sum_{k=1}^K C_k P(\text{escalated to } k)$$

### Complexity
- **ToMe:** $\mathcal{O}(B \cdot (N/2)^2 \cdot d)$ bipartite similarity, reducing downstream attention from $\mathcal{O}(N^2)$ to $\mathcal{O}((N - r)^2)$.
- **Cascade:** Reduces average latency and financial cost by $50\% - 90\%$.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def early_exit_predict(hidden, head_w, head_b, threshold=0.8):
    logits = hidden @ head_w + head_b
    probs = F.softmax(logits, dim=-1)
    confidence, _ = torch.max(probs, dim=-1)
    exit_mask = (confidence >= threshold).to(torch.float32)
    return probs, confidence, exit_mask

def tome_bipartite_merge(x, r_merge=2, token_sizes=None):
    batch, n, dim = x.shape
    if token_sizes is None:
        token_sizes = torch.ones(batch, n, dtype=torch.float32, device=x.device)
    idx_a = torch.arange(0, n, 2, device=x.device)
    idx_b = torch.arange(1, n, 2, device=x.device)
    a, b = x[:, idx_a, :], x[:, idx_b, :]
    sizes_a, sizes_b = token_sizes[:, idx_a], token_sizes[:, idx_b]
    a_norm = a / (torch.norm(a, dim=-1, keepdim=True) + 1e-8)
    b_norm = b / (torch.norm(b, dim=-1, keepdim=True) + 1e-8)
    sim = a_norm @ b_norm.transpose(-1, -2)
    best_sim, best_b = torch.max(sim, dim=-1)
    _, top_a_sub = torch.topk(best_sim, r_merge, dim=-1, largest=True)
    x_out_list, sizes_out_list = [], []
    for b_idx in range(batch):
        cur_b = b[b_idx].clone()
        cur_sizes_b = sizes_b[b_idx].clone()
        a_unmerged, sizes_a_unmerged = [], []
        merged_a_set = set(top_a_sub[b_idx].tolist())
        for a_idx in range(len(idx_a)):
            if a_idx in merged_a_set:
                target_b = best_b[b_idx, a_idx].item()
                s_a, s_b = sizes_a[b_idx, a_idx].item(), cur_sizes_b[target_b].item()
                cur_b[target_b] = (s_b * cur_b[target_b] + s_a * a[b_idx, a_idx]) / (s_a + s_b)
                cur_sizes_b[target_b] = s_a + s_b
            else:
                a_unmerged.append(a[b_idx, a_idx])
                sizes_a_unmerged.append(sizes_a[b_idx, a_idx])
        comb_x = torch.cat([cur_b, torch.stack(a_unmerged)], dim=0) if a_unmerged else cur_b
        comb_s = torch.cat([cur_sizes_b, torch.tensor(sizes_a_unmerged, device=x.device)], dim=0) if a_unmerged else cur_sizes_b
        x_out_list.append(comb_x)
        sizes_out_list.append(comb_s)
    return torch.stack(x_out_list), torch.stack(sizes_out_list)

def frugal_cascade_route(confidences, costs, thresholds):
    batch, num_models = confidences.shape
    routed_idx = torch.full((batch,), num_models - 1, dtype=torch.long, device=confidences.device)
    query_cost = torch.zeros(batch, dtype=torch.float32, device=confidences.device)
    resolved = torch.zeros(batch, dtype=torch.bool, device=confidences.device)
    for m in range(num_models):
        active = ~resolved
        query_cost[active] += costs[m]
        if m < num_models - 1:
            accepted = active & (confidences[:, m] >= thresholds[m])
            routed_idx[accepted] = m
            resolved[accepted] = True
        else:
            routed_idx[active] = m
            resolved[active] = True
    return routed_idx, query_cost

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

probs, conf, exit_mask = early_exit_predict(weights["hidden"], weights["head_w"], weights["head_b"], 0.6)
assert float(torch.max(torch.abs(probs - weights["probs"]))) < 1e-5
assert float(torch.max(torch.abs(conf - weights["conf"]))) < 1e-5
assert float(torch.max(torch.abs(exit_mask - weights["exit_mask"]))) < 1e-5

x_merged, merged_sizes = tome_bipartite_merge(weights["x_tokens"], 2, weights["init_sizes"])
assert float(torch.max(torch.abs(x_merged - weights["x_merged"]))) < 1e-5
assert float(torch.max(torch.abs(merged_sizes - weights["merged_sizes"]))) < 1e-5

routed_idx, total_cost = frugal_cascade_route(weights["confidences"], weights["costs"], weights["thresholds"])
assert float(torch.max(torch.abs(routed_idx.to(torch.float32) - weights["routed_idx"]))) < 1e-5
assert float(torch.max(torch.abs(total_cost - weights["total_cost"]))) < 1e-5
print("All 4 adaptive compute algorithms match golden checkpoints (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_tome = benchmark_fn(lambda: tome_bipartite_merge(weights["x_tokens"], 2), iters=100, name="ToMe Merge (8->6)")
b_cascade = benchmark_fn(lambda: frugal_cascade_route(weights["confidences"], weights["costs"], weights["thresholds"]), iters=100, name="FrugalGPT Route")
print(format_benchmark_table([b_tome, b_cascade]))

baseline_cost = weights["costs"][-1].item() * len(weights["confidences"])
actual_cost = total_cost.sum().item()
savings = (1.0 - actual_cost / baseline_cost) * 100.0
print(f"Cascade savings: {savings:.1f}% cost reduction (${actual_cost:.1f} vs ${baseline_cost:.1f})")

In [ ]:
# Pathological Misuse: Unweighted Token Merging distorts clustered representation mass
t_heavy = torch.ones(1, 16) * 5.0  # Represents a cluster of 5 prior tokens
t_light = torch.zeros(1, 16)       # Represents a single fresh token

# Unweighted average treats both equally: (5.0 + 0.0) / 2 = 2.5 (50% dilution!)
unweighted = (t_heavy + t_light) / 2.0
# Size-weighted average: (5 * 5.0 + 1 * 0.0) / (5 + 1) = 4.167 (preserves mass!)
weighted = (5.0 * t_heavy + 1.0 * t_light) / (5.0 + 1.0)

distortion = torch.norm(unweighted - t_heavy).item()
weighted_err = torch.norm(weighted - t_heavy).item()
assert distortion > 2.0 * weighted_err, f"Unweighted merge must produce greater distortion ({distortion:.2f} vs {weighted_err:.2f})"
print(f"Verified: Unweighted merge error ({distortion:.3f}) >> Size-weighted error ({weighted_err:.3f})")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Heterogeneous Input Efficiency**: Uniform compute over-allocates resources to easy queries; early exit cuts latency by $1.5\times-3\times$ with negligible accuracy loss.
2. **ToMe Preserves Attention Mass**: Tracking token cluster size $s$ is mathematically required during bipartite fusion to prevent representational dilution.
3. **Cascade Economics**: Routing $75\%$ of traffic to sub-cent lightweight models yields dramatic cost savings ($>60\%$) while preserving frontier model benchmark parity.

---
### Cross-Domain Generalisation Matrix
- **Vision Transformers:** ToMe drops $30\%-50\%$ of background patches in ViT/DeiT without fine-tuning.
- **Diffusion Models:** ToMe merges redundant spatial latents during early diffusion timesteps, halving generation time.
- **Enterprise LLM APIs:** FrugalGPT dynamically balances SLAs, budgets, and quality constraints across vendor APIs.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.md](../../../markdown/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.md)
- **PyTorch 2.x Module:** [01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.py](../../../pytorch2/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.py)
- **Functional JAX Module:** [01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.py](../../../jax/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.py)
- **Native Rust Candle Track:** [01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.rs](../../../candle/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.rs)
- **Hardware Kernel (CPU / WGSL):** [01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.rs](../../../kernels/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.rs)
- **Golden SafeTensors:** [01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.safetensors](../../../weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Knowledge Distillation Logit Feature and Minillm](../01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 02. KV Cache Paged Radix H2O Snapkv and Yoco](02_kv_cache_paged_radix_h2o_snapkv_and_yoco.ipynb) &rarr;